# 02 — Data Preprocessing (Week 3)

Pipeline (logic lives in `src/idx_avm/` so notebooks, scripts and tests share one implementation):

1. Load → single-family filter
2. Row-level cleaning with a logged count at every step (best-practices §02)
3. Row-wise feature construction (property age, cyclical month) + leakage audit
4. Chronological split: **test = most recent complete month**, train = the **X** months right before it
5. Tune **X** on validation months only (rolling origin), never on the test month
6. ClosePrice outlier cutoffs (0.5 / 99.5 pct) learned on **train only**, frozen and applied to test
7. Imputation / encoding / scaling inside a scikit-learn `ColumnTransformer`, fit on train only
8. Save cleaned train/test CSVs + logs to `data/processed/`

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from idx_avm import config

pd.set_option("display.max_columns", 80)
sns.set_theme(style="whitegrid")
np.random.seed(config.RANDOM_SEED)
print("Raw data dir:", config.RAW_DIR)

## 1. Load and filter

In [ ]:
from idx_avm.load import load_raw, filter_single_family

raw = load_raw()
sfr = filter_single_family(raw)
print(f"{len(raw):,} raw rows -> {len(sfr):,} single-family residential")

## 2. Cleaning (logged)

Rules and why:

| Rule | Action | Reason |
|---|---|---|
| ClosePrice missing or ≤ 0 | drop | invalid target; target is never imputed |
| exact duplicates / repeated `ListingKey` / same address + CloseDate | drop (keep latest) | same sale in several monthly files |
| CloseDate before ListingContractDate | drop | impossible; ListingContractDate used **only** for this QA check |
| LivingArea ≤ 0, or < 70 sq ft per bedroom | drop | data-entry error |
| Bedrooms / bathrooms outside [0, 20] | drop | data-entry error |
| YearBuilt < 1850 or after the sale year | set NaN → imputed | keep the sale, distrust the field |
| LotSizeSquareFeet missing but LotSizeAcres present | backfill × 43,560 | same fact, other unit |
| LotSizeSquareFeet ≤ 0, coordinates outside CA | set NaN → imputed | keep the sale, distrust the field |

In [ ]:
from idx_avm.clean import clean

df, log = clean(sfr)
log_df = log.to_frame()
display(log_df)
print(f"Kept {len(df):,} of {len(sfr):,} rows ({len(df) / len(sfr):.1%})")

## 3. Row-wise features and the feature whitelist

Only property characteristics and location — information available for any address at prediction time.

In [ ]:
from idx_avm.features import add_base_features, feature_columns
from idx_avm.split import add_close_month, complete_months

df = add_close_month(add_base_features(df))
groups = feature_columns(df)          # raises if any leaky column slipped in
for name, cols in groups.items():
    print(f"{name:9s} {cols}")
missing_whitelist = [c for c in config.NUMERIC_FEATURES + config.BOOLEAN_FEATURES
                     + config.LOW_CARD_CATEGORICAL + config.HIGH_CARD_CATEGORICAL if c not in df.columns]
print("Whitelisted but absent from this extract:", missing_whitelist)

### Missingness in the chosen features

Numeric features get median imputation **plus a missing-indicator column**; categoricals get a `Missing` level.

In [ ]:
feat_cols = [c for cols in groups.values() for c in cols]
df[feat_cols].isna().mean().sort_values(ascending=False).rename("pct_missing").to_frame().style.format("{:.1%}")

## 4. Months available

A trailing partial month (extract taken mid-month) is excluded so the test month is complete.

In [ ]:
months = complete_months(df)
dropped = sorted(set(df["CloseMonth"].unique()) - set(months))
df = df[df["CloseMonth"].isin(months)]
print("Complete months:", [str(m) for m in months])
print("Dropped partial month(s):", [str(m) for m in dropped])
TEST_MONTH = months[-1]
print("Test month:", TEST_MONTH)

## 5. Tune the training window X

For each candidate X, train a linear baseline (log target, same preprocessing pipeline) on the X months before a
**validation** month and score that month. Repeat over the 3 months before the test month and average, so the
choice isn't driven by one lucky month. The test month is never used here.

Trade-off: a longer window gives more rows, but older sales reflect a different price level.

In [ ]:
from idx_avm.window import tune_training_window

max_x = len(months) - 2
candidates = [x for x in [1, 2, 3, 4, 5, 6, 9, 12, 18, 24] if x <= max_x]
n_origins = min(3, len(months) - 2)
by_origin, summary = tune_training_window(df, months, candidates, n_origins=n_origins)
display(by_origin.pivot(index="X", columns="val_month", values="MdAPE").round(2))
display(summary.round(3))
BEST_X = int(summary.index[0])
print(f"Chosen training window: X = {BEST_X} months (lowest mean validation MdAPE)")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for vm, g in by_origin.groupby("val_month"):
    ax.plot(g["X"], g["MdAPE"], marker="o", alpha=0.5, label=f"val {vm}")
ax.plot(summary.sort_index().index, summary.sort_index()["MdAPE"], marker="o", color="k", lw=2, label="mean")
ax.set(xlabel="training window X (months)", ylabel="validation MdAPE (%)", title="Choosing X")
ax.legend(); plt.show()

## 6. Final split and train-only outlier thresholds

In [ ]:
from idx_avm.split import window_split, apply_outlier_filter

split, thresholds, stats = apply_outlier_filter(window_split(df, TEST_MONTH, BEST_X))
print("Train months:", [str(m) for m in split.train_months], "| Test month:", split.test_month)
print(f"ClosePrice cutoffs learned on train: ${stats['lower']:,.0f} - ${stats['upper']:,.0f}")
print(f"Removed by outlier filter: train {stats['train_removed']:,}, test {stats['test_removed']:,}")
print(f"Final sizes: train {len(split.train):,} | test {len(split.test):,}")
assert split.train["CloseDate"].max() < split.test["CloseDate"].min(), "split is not chronological"


## 7. Preprocessing pipeline (fit on train only)

The Week 4 baseline will reuse exactly this pipeline via `build_model_pipeline`.

In [ ]:
from idx_avm.preprocess import build_preprocessor, feature_frame

groups = feature_columns(split.train)
pre = build_preprocessor(groups)
X_train = pre.fit_transform(feature_frame(split.train, groups), np.log(split.train[config.TARGET]))
X_test = pre.transform(feature_frame(split.test, groups))
names = pre.get_feature_names_out()
print("Encoded matrix:", X_train.shape, "train /", X_test.shape, "test")
pd.DataFrame(X_train, columns=names).describe().T[["mean", "std", "min", "max"]].round(3)

## 8. Save cleaned CSVs and metadata

The cleaned CSVs hold the *un-encoded* features (encoding is learned inside the model pipeline at fit time). `data/` is git-ignored.

In [ ]:
import json

out = config.PROCESSED_DIR
out.mkdir(parents=True, exist_ok=True)
keep = ["ListingKey"] if "ListingKey" in df.columns else []
cols = keep + feat_cols + [config.DATE_COL, "CloseMonth", config.TARGET]
split.train[cols].to_csv(out / "train_clean.csv", index=False)
split.test[cols].to_csv(out / "test_clean.csv", index=False)
log_df.to_csv(out / "cleaning_log.csv", index=False)
summary.to_csv(out / "window_tuning_summary.csv")
meta = {
    "source_files": sorted(raw["SourceFile"].unique()),
    "train_months": [str(m) for m in split.train_months],
    "test_month": str(split.test_month),
    "chosen_window_months": BEST_X,
    "outlier_thresholds": stats,
    "n_train": len(split.train), "n_test": len(split.test),
    "feature_groups": groups,
}
(out / "split_metadata.json").write_text(json.dumps(meta, indent=2, default=str))
print("Saved to", out); print(json.dumps(meta, indent=2, default=str)[:1500])

## 9. Summary (fill in from the run)

- Rows: raw → single-family → cleaned → train / test
- Biggest cleaning steps and their % (see `cleaning_log.csv`)
- Chosen X and how flat/steep the validation curve was
- Outlier cutoffs (train-learned)
- Leakage: list-price family, DOM, contract dates excluded from inputs; ListingContractDate used only for QA